In [1]:
import os
import yaml
import warnings; warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from byaldi import RAGMultiModalModel
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

In [2]:
load_dotenv(dotenv_path="../.env") 

with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

env_var_name = config['api_keys']['google_gemini_env_name']
gemini_api_key = os.getenv(env_var_name)

if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    print("✅ Gemini API 키가 성공적으로 로드되었습니다.")
else:
    print("❌ API 키를 찾을 수 없습니다. .env 파일을 확인해주세요.")

✅ Gemini API 키가 성공적으로 로드되었습니다.


In [3]:
print("⏳ ColPali 비전 모델을 불러오는 중...")
vision_retriever = RAGMultiModalModel.from_pretrained("vidore/colpali-v1.2")

⏳ ColPali 비전 모델을 불러오는 중...
Verbosity is set to 1 (active). Pass verbose=0 to make quieter.


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 434.93it/s]
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [4]:
# Baseline의 TextLoader + Chunking + Chroma DB 생성을 하나로 합친 과정입니다.
pdf_path = "../data/raw/samsung_report.pdf"
index_name = "samsung_vision_index"

print(f"📚 PDF 이미지 벡터 변환 및 인덱싱 시작... (시간이 소요될 수 있습니다)")

# PDF를 이미지로 잘라 벡터 DB(ColPali 인덱스)를 구축합니다.
vision_retriever.index(
    input_path=pdf_path,
    index_name=index_name,
    store_collection_with_index=True,
    overwrite=True
)
print("✅ ColPali Vision DB 구축 완료!")

📚 PDF 이미지 벡터 변환 및 인덱싱 시작... (시간이 소요될 수 있습니다)
Added page 1 of document 0 to index.
Added page 2 of document 0 to index.
Added page 3 of document 0 to index.
Added page 4 of document 0 to index.
Added page 5 of document 0 to index.
Added page 6 of document 0 to index.
Added page 7 of document 0 to index.
Added page 8 of document 0 to index.
Added page 9 of document 0 to index.
Added page 10 of document 0 to index.
Added page 11 of document 0 to index.
Added page 12 of document 0 to index.
Added page 13 of document 0 to index.
Added page 14 of document 0 to index.
Added page 15 of document 0 to index.
Added page 16 of document 0 to index.
Added page 17 of document 0 to index.
Added page 18 of document 0 to index.
Added page 19 of document 0 to index.
Added page 20 of document 0 to index.
Added page 21 of document 0 to index.
Added page 22 of document 0 to index.
Added page 23 of document 0 to index.
Added page 24 of document 0 to index.
Added page 25 of document 0 to index.
Added page 26 

In [10]:
print("\n" + "="*50)
print("🧐 AI에게 어려운 질문을 던져봅시다 (비전 데이터 검색 테스트)")

# Baseline과 완전히 동일한 질문으로 비교(Ablation Study)합니다.
query = "2024년 제56기 삼성전자의 당기순이익은 얼마인가요?"
print(f"질문: {query}\n")

print("🔍 ColPali로 질문과 가장 유사한 '페이지 이미지'를 찾습니다...\n")


🧐 AI에게 어려운 질문을 던져봅시다 (비전 데이터 검색 테스트)
질문: 2024년 제55기 삼성전자의 당기순이익은 얼마인가요?

🔍 ColPali로 질문과 가장 유사한 '페이지 이미지'를 찾습니다...



In [11]:
# Baseline의 retriever.invoke() 와 같은 역할입니다.
# 가장 관련성 높은 1개의 페이지 이미지를 찾아 Base64 문자열로 가져옵니다.
results = vision_retriever.search(query, k=1)

best_page_base64 = results[0].base64
best_page_num = results[0].page_num

print(f"🎯 검색 완료: {best_page_num}페이지가 가장 관련성이 높습니다.")

🎯 검색 완료: 449페이지가 가장 관련성이 높습니다.


In [12]:
# Baseline과 동일하게 Gemini 2.5 Flash를 로드합니다.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [13]:
# LangChain의 멀티모달 프롬프트 방식을 사용하여 텍스트 지시문과 이미지를 함께 묶어줍니다.
message = HumanMessage(
    content=[
        {
            "type": "text", 
            "text": f"당신은 꼼꼼한 금융 데이터 분석가입니다.\n아래 제공된 [문서 이미지]를 직접 눈으로 확인하고 사용자의 [질문]에 답변해주세요.\n표 데이터가 헷갈린다면 유추하지 말고 '명확한 수치를 찾기 어렵습니다.'라고 답변하세요.\n\n[질문]\n{query}"
        },
        {
            "type": "image_url", 
            "image_url": {"url": f"data:image/jpeg;base64,{best_page_base64}"}
        }
    ]
)

In [14]:
# Chain 대신 메시지 객체를 바로 invoke 하여 답변을 생성합니다.
response = llm.invoke([message])

print("=" * 50)
print("🤖 최종 AI 답변 (Vision RAG):")
print(response.content)
print("=" * 50)

🤖 최종 AI 답변 (Vision RAG):
제공된 [문서 이미지]는 삼성전자의 대주주 등과의 거래내용, 특히 해외 종속법인에 대한 채무보증 현황을 담고 있습니다.

이 문서에는 2024년 제55기 삼성전자의 당기순이익에 대한 정보가 포함되어 있지 않습니다. 당기순이익은 손익계산서와 같은 재무제표에서 확인할 수 있는 항목입니다.
